# Train Logistic Regression with Minibatches

This tutorial shows how to train an incremental logistic-regression classifier
from Atlas expression minibatches. It is the simplest supervised example of
using scAtlasPy as a data-access layer for custom model development.

The example uses scikit-learn's `SGDClassifier` with logistic loss. Expression
data and labels are read in dense minibatches, while the full cell-by-gene
matrix remains in the Atlas.

By the end of this tutorial, you will be able to:

- define a labeled training population in `obs`;
- retrieve expression minibatches together with matching labels;
- train a classifier incrementally with `partial_fit()`;
- make randomized multi-pass training scans without loading the full matrix;
- evaluate the training pipeline in a deterministic streaming pass;
- save the fitted model for later full-Atlas prediction.

## Before You Begin

This tutorial assumes that:

- quality control and preprocessing have been completed;
- `obs.cell_type_manual` contains the target labels;
- scaled expression values are available in `data_scale`;
- the selected dense minibatches fit in memory.

Open the existing Atlas:

In [1]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import scatlaspy as sap

sap.set_progress(False)

os.chdir("/home/hanxu/scatlas-benchmarking")

atlas_path = Path("tmp/tutorials/basic_pbmc3k/pbmc3k_basic.sasql")

if not atlas_path.is_file():
    raise FileNotFoundError(
        f"Atlas database not found: {atlas_path}. Run the basic exploration tutorial first."
    )

atlas = sap.Atlas(atlas_path)

/home/hanxu/anaconda3/envs/scatlas-benchmarking/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


```{note}
This tutorial uses known cell-type labels only as an example supervised target.
The same pattern can be applied to another categorical outcome stored in
`obs`.
```

## 1. Define the Labeled Training Population

Use the existing cell filter and keep only cells with a non-missing training
label. The filtered population is inspected with the `get_obs_df()` API; no
manual SQL is needed.

In [2]:
obs = atlas.get_obs_df(columns=["filter_cells", "cell_type_manual"])

training_obs = obs.loc[
    obs["filter_cells"].fillna(False) & obs["cell_type_manual"].notna()
].copy()

if training_obs.empty:
    raise ValueError("No labeled cells are available for training.")


Inspect the number of labeled cells in each class:



In [3]:
class_counts = (
    training_obs["cell_type_manual"]
    .value_counts()
    .rename_axis("cell_type_manual")
    .reset_index(name="n_cells")
)

class_counts

,cell_type_manual,n_cells
0,CD4 T,1149
1,CD14+ Monocytes,495
2,B,354
3,CD8 T,353
4,FCGR3A+ Monocytes,172
5,NK,143
6,Dendritic,34


Confirm that the selected population contains at least two classes and that
each class contains enough cells for the intended analysis.

```{important}
The training read index must select the same cell population that carries the
labels used by the model. If your atlas contains unlabeled filtered cells, add a
Boolean obs column for the labeled subset before building the read index.
```

## 2. Build the Training Read Index

Build a read index using the filtered cells, filtered genes, highly variable
genes, and scaled expression values:

In [4]:
atlas.build_read_index(
    cell_condition="filter_cells",
    gene_condition="filter_genes",
    use_hvg=True,
    use_data="data_scale",
)

The index table required for minibatch expression matrix reading has been rebuilt. Please rerun PCA, clustering, and other operations that depend on this table!


The model input now consists of:

- cells selected by `filter_cells`;
- genes selected by `filter_genes`;
- genes marked as highly variable;
- scaled expression values stored in `data_scale`.

```{important}
The trained model depends on the exact gene set, gene order, and expression
representation defined by this read index. Record this information alongside
the fitted model.
```

Rebuilding the read index changes the active data view used by subsequent
streaming operations. It does not change the stored expression values, but
later computations should use a read index appropriate for their own purpose.

## 3. Prepare the Label Encoder

Read the labels from the active read index once to fit the encoder. During
training and evaluation, labels will be retrieved directly with each minibatch
using `get_obs_col="cell_type_manual"`.

In [5]:
label_df = atlas.get_obs_df(columns=["filter_cell_id", "cell_type_manual"])
label_df = (
    label_df.dropna(subset=["filter_cell_id"])
    .sort_values("filter_cell_id")
    .reset_index(drop=True)
)


Verify that no selected cell has a missing label:



In [6]:
if label_df.empty:
    raise ValueError("The current read index contains no cells.")

if label_df["cell_type_manual"].isna().any():
    raise ValueError(
        "The current read index contains cells without training labels. "
        "Create a labeled-cell obs column and rebuild the read index."
    )

if label_df["filter_cell_id"].duplicated().any():
    raise ValueError(
        "The current read index contains duplicated filter_cell_id values."
    )


Encode text labels as consecutive integers:



In [7]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoder.fit(label_df["cell_type_manual"].to_numpy())

classes = np.arange(len(label_encoder.classes_))
n_training_cells = len(label_df)

print(f"Training cells: {n_training_cells:,}")
print(f"Classes: {len(classes):,}")
print(label_encoder.classes_)

Training cells: 2,700
Classes: 7
['B' 'CD14+ Monocytes' 'CD4 T' 'CD8 T' 'Dendritic' 'FCGR3A+ Monocytes'
 'NK']



Check that classification is possible:



In [8]:
if len(classes) < 2:
    raise ValueError(
        "At least two label classes are required for classification."
    )



The encoded labels use less memory and can later be converted back to their
original names with `label_encoder.inverse_transform()`.

## 4. Configure the Classifier

Create an incremental linear classifier with logistic loss:



In [9]:
from sklearn.linear_model import SGDClassifier

model = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=1e-4,
    random_state=42,
)


`partial_fit()` updates the existing model from one minibatch at a time. The
complete class list must be supplied during the first update because the first
minibatch may not contain every class.

The scaled input used in this tutorial is appropriate for gradient-based
optimization. If another expression field is used, consider whether its scale
is suitable for the selected model.

## 5. Train in Label-Aware Minibatches

Set the minibatch size, the number of passes, and the shuffle-buffer size:

In [10]:
batch_size = 2048
n_epochs = 3
buffer_batch_num = 5
batches_per_epoch = int(np.ceil(n_training_cells / batch_size))

Training uses `pass_mode="multi-pass"`, which repeatedly scans the active read
index and shuffles dense minibatches through a bounded buffer. Because
`get_obs_col="cell_type_manual"` is supplied, each yielded batch contains both
`X` and the matching labels.

In [11]:
first_update = True

for epoch in range(1, n_epochs + 1):
    epoch_cells = 0

    for batch_id, batch in enumerate(
        atlas.get_minibatch_dense(
            pass_mode="multi-pass",
            batch_size=batch_size,
            buffer_batch_num=buffer_batch_num,
            max_batches=batches_per_epoch,
            get_obs_col="cell_type_manual",
        ),
        start=1,
    ):
        X_batch = np.asarray(batch["X"], dtype=np.float32)
        label_values = np.asarray(batch["cell_type_manual"], dtype=object)

        if X_batch.ndim != 2:
            raise ValueError("Each expression minibatch must be a two-dimensional matrix.")

        if pd.isna(label_values).any():
            raise ValueError("A training minibatch contains missing labels.")

        y_batch = label_encoder.transform(label_values)

        if first_update:
            model.partial_fit(X_batch, y_batch, classes=classes)
            first_update = False
        else:
            model.partial_fit(X_batch, y_batch)

        epoch_cells += X_batch.shape[0]

        if batch_id == 1 or batch_id % 50 == 0:
            print(
                f"Epoch {epoch}/{n_epochs}, "
                f"batch {batch_id}: "
                f"processed {epoch_cells:,} cells"
            )

    print(f"Completed epoch {epoch}/{n_epochs}: {epoch_cells:,} cells")

Epoch 1/3, batch 1: processed 2,048 cells


Completed epoch 1/3: 2,700 cells
Epoch 2/3, batch 1: processed 2,048 cells
Completed epoch 2/3: 2,700 cells


Epoch 3/3, batch 1: processed 2,048 cells
Completed epoch 3/3: 2,700 cells


Only the current expression minibatch, its matching labels, and the model
parameters need to reside in memory.

```{note}
`multi-pass` mode should usually be used with `max_batches` so that training has
a clear stopping rule. Increasing `buffer_batch_num` improves shuffling across
nearby minibatches but increases the dense buffer memory footprint.
```

## 6. Check the Training Pipeline

Run a deterministic single pass and calculate predictions without retaining all
expression data or predictions:

In [12]:
correct = 0
total = 0

for batch in atlas.get_minibatch_dense(
    pass_mode="single-pass",
    batch_size=batch_size,
    get_obs_col="cell_type_manual",
):
    X_batch = np.asarray(batch["X"], dtype=np.float32)
    label_values = np.asarray(batch["cell_type_manual"], dtype=object)

    if pd.isna(label_values).any():
        raise ValueError("An evaluation minibatch contains missing labels.")

    y_batch = label_encoder.transform(label_values)
    pred = model.predict(X_batch)

    correct += int(np.count_nonzero(pred == y_batch))
    total += X_batch.shape[0]

if total == 0:
    raise ValueError("The evaluation stream contained no cells.")

training_accuracy = correct / total

print(f"Training-set accuracy: {training_accuracy:.3f}")

Training-set accuracy: 1.000


```{warning}
Training-set accuracy checks that expression values, labels, and model outputs
are aligned correctly. It is not an unbiased estimate of predictive
performance because the same cells were used for fitting and evaluation.
```

## 7. Evaluate on Held-out Cells

For a meaningful evaluation, define non-overlapping training and test
populations in `obs`, build the read index for each population, and retrieve
labels with `get_obs_col` during the corresponding streaming pass.

```{tip}
Choose a validation split that reflects the intended application. For example,
holding out complete donors, samples, studies, or technologies can be more
informative than randomly holding out individual cells when the goal is
generalization across biological or technical contexts.
```

Accuracy may also be misleading when classes are highly imbalanced. Consider
class-specific recall, precision, F1 scores, and a confusion matrix when
evaluating the final model.

The same label-aware pattern can be used in other supervised minibatch loops:

```python
for batch in atlas.get_minibatch_dense(
    pass_mode="multi-pass",
    batch_size=batch_size,
    max_batches=100,
    get_obs_col="cell_type_manual",
):
    X_batch = batch["X"]
    y_batch = label_encoder.transform(batch["cell_type_manual"])
    model.partial_fit(X_batch, y_batch)
```


## 8. Inspect the Fitted Model

Check the fitted coefficient dimensions:

In [13]:
print("Classes:", model.classes_)
print("Coefficient shape:", model.coef_.shape)
print("Intercept shape:", model.intercept_.shape)


Classes: [0 1 2 3 4 5 6]
Coefficient shape: (7, 2000)
Intercept shape: (7,)



For multiclass classification, the coefficient matrix contains one row per
class and one column per selected gene.

Convert encoded predictions back to their original labels with:



In [14]:
predicted_names = label_encoder.inverse_transform(
    model.predict(X_batch)
)



Interpret model coefficients only when the feature identities, feature order,
and preprocessing representation have been retained.

## 10. Save the Model

Save the classifier and label encoder for later use:



In [15]:
from pathlib import Path

import joblib

output_path = Path("tmp/tutorials/advanced/logistic_regression_model.joblib")
output_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(
    {
        "model": model,
        "label_encoder": label_encoder,
    },
    output_path,
)

print(f"Saved model to {output_path}")

Saved model to tmp/tutorials/advanced/logistic_regression_model.joblib


Also retain:

- the selected gene names in their exact read-index order;
- the expression field;
- normalization and scaling settings;
- the cell-selection definition;
- model hyperparameters;
- the scAtlasPy and scikit-learn versions.

The model cannot be applied safely to another Atlas unless its input features
match the training features exactly.

## Limitations of This Example

This tutorial demonstrates the data-access and label-alignment pattern rather
than a complete cell-type classification workflow.

It does not include:

- hyperparameter selection;
- class-imbalance correction;
- donor- or sample-aware validation;
- probability calibration;
- feature interpretation;
- automated early stopping.

These decisions should be adapted to the biological question and intended
generalization setting.

## Close the Atlas

Close the database connection when this tutorial is complete. This releases
the DuckDB file lock so the same `.sasql` Atlas can be opened by another
notebook or Python session.


In [16]:
atlas.close()


## Next Steps

Continue with {doc}`train-pytorch-model-with-minibatches` for a neural-network
implementation of supervised minibatch training.

See {doc}`apply-model-to-full-atlas` to apply the fitted classifier to every
cell selected by a compatible read index.